# Étape 3 — Fine-tuning QLoRA et évaluation

Ce notebook :
1. découpe `offres_propres.jsonl` en **train** et **test** (une seule fois, graine fixe) ;
2. mesure le **modèle de base** sur le test ;
3. entraîne un adaptateur **QLoRA** ;
4. mesure le **modèle fine-tuné** sur le même test ;
5. sauvegarde le tableau comparatif et pousse l'adaptateur sur Hugging Face (en privé).

**Avant de lancer :** Exécution → Modifier le type d'exécution → GPU T4.
Durée totale estimée : 30 à 50 minutes.

In [ ]:
# 1. Installation + Google Drive
!pip install -q transformers accelerate peft bitsandbytes datasets pydantic pandas

from google.colab import drive
drive.mount('/content/drive')

import os
DOSSIER = '/content/drive/MyDrive/job-offer-extractor/data'
RESULTATS = '/content/drive/MyDrive/job-offer-extractor/resultats'
os.makedirs(RESULTATS, exist_ok=True)
SOURCE = f'{DOSSIER}/offres_propres.jsonl'
TRAIN = f'{DOSSIER}/train.jsonl'
TEST = f'{DOSSIER}/test.jsonl'

In [ ]:
# 2. Connexion Hugging Face
from huggingface_hub import login
login()

In [ ]:
# 3. Schéma + prompt système (identiques à l'étape 2, avec SIVP)
import json
from typing import Literal, Optional
from pydantic import BaseModel, ValidationError

class Offre(BaseModel):
    poste: str
    entreprise: Optional[str]
    ville: Optional[str]
    type_contrat: Optional[Literal['PFE', 'stage', 'CDI', 'CDD', 'SIVP', 'freelance']]
    mode_travail: Optional[Literal['présentiel', 'hybride', 'remote']]
    duree_mois: Optional[float]
    competences: list[str]
    remuneration: Optional[str]
    langue_offre: Literal['fr', 'en', 'ar']

SCHEMA = json.dumps(Offre.model_json_schema(), ensure_ascii=False)

REGLES = '''Règles :
- type_contrat = "PFE" si l'offre parle de PFE ou de fin d'études, sinon "stage" pour un stage. Si plusieurs types sont cités, prendre le premier.
- remuneration = le montant s'il est donné ; "rémunéré" si l'offre dit seulement rémunéré ; une prime n'est pas un salaire ; "Confidentiel" = null.
- competences = uniquement des compétences techniques ou métier (outils, langages, domaines). Jamais d'années d'expérience, de diplôme, de tâches ni de qualités humaines.
- langue_offre = UNE seule valeur, la langue dominante.
- ville et entreprise sont deux champs séparés.
- Si une information est absente, mets null. Ne devine jamais.'''

SYSTEM_PROMPT = (
    "Tu extrais les informations d'une offre d'emploi ou de stage. "
    'Réponds UNIQUEMENT avec un JSON valide respectant ce schéma :\n'
    + SCHEMA + '\n\n' + REGLES
)

## Découpage train / test

Le test est tiré **une seule fois** (graine 42) puis enregistré sur Drive. Si `test.jsonl` existe déjà, on le réutilise : toutes tes comparaisons futures (v1, v2…) se feront sur exactement les mêmes offres.

In [ ]:
# 4. Découpage stratifié par langue (environ 20 % en test)
import random
from collections import Counter, defaultdict

def lire(chemin):
    with open(chemin, encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

def ecrire(chemin, lignes):
    with open(chemin, 'w', encoding='utf-8') as f:
        for l in lignes:
            f.write(json.dumps(l, ensure_ascii=False) + '\n')

if os.path.exists(TEST):
    train, test = lire(TRAIN), lire(TEST)
    print('Découpage existant réutilisé.')
else:
    donnees = lire(SOURCE)
    random.seed(42)
    par_langue = defaultdict(list)
    for d in donnees:
        par_langue[d['extraction']['langue_offre']].append(d)
    train, test = [], []
    for langue, groupe in par_langue.items():
        random.shuffle(groupe)
        n_test = max(2, round(len(groupe) * 0.2))
        test += groupe[:n_test]
        train += groupe[n_test:]
    # au moins 2 PFE dans le test, pour mesurer la catégorie qui nous intéresse
    pfe_test = [d for d in test if d['extraction']['type_contrat'] == 'PFE']
    pfe_train = [d for d in train if d['extraction']['type_contrat'] == 'PFE']
    while len(pfe_test) < 2 and pfe_train:
        d = pfe_train.pop()
        train.remove(d); test.append(d); pfe_test.append(d)
    ecrire(TRAIN, train)
    ecrire(TEST, test)
    print('Nouveau découpage enregistré.')

for nom, part in [('Train', train), ('Test', test)]:
    print(f'{nom} : {len(part)} offres |',
          'langues', dict(Counter(d['extraction']['langue_offre'] for d in part)), '|',
          'contrats', dict(Counter(d['extraction']['type_contrat'] for d in part)))

In [ ]:
# 5. Charger le modèle de base en 4 bits (QLoRA)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = 'right'

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map='auto')
print('Modèle chargé. Mémoire GPU utilisée :', round(torch.cuda.memory_allocated() / 1e9, 2), 'Go')

## Évaluation

Pour chaque offre du test, on mesure :
- **JSON valide** et **schéma respecté** ;
- **exactitude par champ** (poste, entreprise, ville, contrat, mode, durée, rémunération, langue) ;
- **F1 des compétences** : précision (pas de compétence inventée) et rappel (pas de compétence oubliée).

In [ ]:
# 6. Fonctions de génération et d'évaluation
import pandas as pd

def nettoyer(texte):
    return texte.strip().removeprefix('```json').removesuffix('```').strip()

@torch.no_grad()
def extraire(offre, max_new_tokens=700):
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': offre}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors='pt', return_dict=True
    ).to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

CHAMPS = ['poste', 'entreprise', 'ville', 'type_contrat', 'mode_travail',
          'duree_mois', 'remuneration', 'langue_offre']

def norm(v):
    if v is None:
        return None
    if isinstance(v, (int, float)):
        return round(float(v), 2)
    return ' '.join(str(v).lower().split())

def egal(champ, pred, vrai):
    if champ == 'duree_mois' and pred is not None and vrai is not None:
        try:
            return abs(float(pred) - float(vrai)) < 0.05
        except (TypeError, ValueError):
            return False
    return norm(pred) == norm(vrai)

def evaluer(nom):
    lignes, tp, fp, fn = [], 0, 0, 0
    model.eval()
    for i, ex in enumerate(test):
        brut = extraire(ex['offre'])
        vrai = ex['extraction']
        ligne = {'url': ex.get('url'), 'langue': vrai['langue_offre'], 'sortie_brute': brut}
        try:
            pred = json.loads(nettoyer(brut))
            ligne['json_valide'] = True
        except json.JSONDecodeError:
            pred = {}
            ligne['json_valide'] = False
        try:
            Offre.model_validate(pred)
            ligne['schema_ok'] = True
        except ValidationError:
            ligne['schema_ok'] = False
        for c in CHAMPS:
            ligne[c] = egal(c, pred.get(c), vrai.get(c))
        p = {norm(x) for x in (pred.get('competences') or []) if isinstance(x, str)}
        v = {norm(x) for x in vrai['competences']}
        tp += len(p & v); fp += len(p - v); fn += len(v - p)
        lignes.append(ligne)
        print(f'{nom} {i+1}/{len(test)}', end='\r')
    df = pd.DataFrame(lignes)
    prec = tp / (tp + fp) if tp + fp else 0
    rapp = tp / (tp + fn) if tp + fn else 0
    f1 = 2 * prec * rapp / (prec + rapp) if prec + rapp else 0
    resume = {'JSON valide': df['json_valide'].mean(), 'Schéma respecté': df['schema_ok'].mean()}
    resume.update({c: df[c].mean() for c in CHAMPS})
    resume.update({'Compétences précision': prec, 'Compétences rappel': rapp, 'Compétences F1': f1})
    df.to_csv(f'{RESULTATS}/details_{nom}.csv', index=False)
    print()
    return pd.Series(resume, name=nom).round(3)

In [ ]:
# 7. Baseline : le modèle de base sur le test (10 à 15 min)
res_base = evaluer('base')
res_base

## Fine-tuning QLoRA

Le modèle apprend uniquement à produire **le JSON** : les tokens du prompt système et de l'offre sont masqués dans la perte (label `-100`).

In [ ]:
# 8. Préparer les exemples d'entraînement
from datasets import Dataset

MAX_LEN = 2560

def construire(ex):
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': ex['offre']}]
    prompt_ids = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True, return_dict=True
    )['input_ids']
    reponse = json.dumps(ex['extraction'], ensure_ascii=False) + tokenizer.eos_token
    reponse_ids = tokenizer(reponse, add_special_tokens=False)['input_ids']
    input_ids = (prompt_ids + reponse_ids)[:MAX_LEN]
    labels = ([-100] * len(prompt_ids) + reponse_ids)[:MAX_LEN]
    return {'input_ids': input_ids, 'attention_mask': [1] * len(input_ids), 'labels': labels}

train_ds = Dataset.from_list([construire(ex) for ex in train])
longueurs = [len(x) for x in train_ds['input_ids']]
print(len(train_ds), 'exemples | longueur max', max(longueurs), 'tokens')
assert max(longueurs) < MAX_LEN, 'Un exemple dépasse MAX_LEN : augmente MAX_LEN'

In [ ]:
# 9. Ajouter les adaptateurs LoRA
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
# 10. Entraînement (10 à 20 min)
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

args = TrainingArguments(
    output_dir='/content/qlora-offres',
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_steps=2,
    fp16=True,
    optim='paged_adamw_8bit',
    gradient_checkpointing=True,
    logging_steps=5,
    save_strategy='no',
    report_to='none',
    seed=42,
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100),
)
model.config.use_cache = False
trainer.train()
model.config.use_cache = True

In [ ]:
# 11. Courbe de perte (à mettre dans ton README)
import matplotlib.pyplot as plt

logs = [l for l in trainer.state.log_history if 'loss' in l]
plt.plot([l['step'] for l in logs], [l['loss'] for l in logs], marker='o')
plt.xlabel('Étape'); plt.ylabel('Perte'); plt.title('Perte d’entraînement QLoRA')
plt.grid(alpha=0.3)
plt.savefig(f'{RESULTATS}/courbe_perte.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 12. Évaluation du modèle fine-tuné (10 à 15 min)
res_ft = evaluer('finetune_v1')

In [ ]:
# 13. Tableau comparatif
comparaison = pd.concat([res_base, res_ft], axis=1)
comparaison['gain'] = (comparaison['finetune_v1'] - comparaison['base']).round(3)
comparaison.to_csv(f'{RESULTATS}/comparaison_v1.csv')
print(f'Test : {len(test)} offres')
comparaison

In [ ]:
# 14. Exemple côte à côte sur une offre PFE du test
ex = next((d for d in test if d['extraction']['type_contrat'] == 'PFE'), test[0])
print('ATTENDU :')
print(json.dumps(ex['extraction'], ensure_ascii=False, indent=2))
print('\nMODÈLE FINE-TUNÉ :')
print(extraire(ex['offre']))

In [ ]:
d = pd.read_csv(f'{RESULTATS}/details_finetune_v1.csv')
for _, r in d[~d['json_valide']].iterrows():
    print('URL :', r['url'], '| langue :', r['langue'])
    print('Longueur sortie :', len(r['sortie_brute']), 'caractères')
    print('DÉBUT :', r['sortie_brute'][:300])
    print('FIN :', r['sortie_brute'][-300:])
    print('-' * 60)

In [ ]:
# 15. Sauvegarder l'adaptateur sur Drive et sur Hugging Face (privé)
ADAPTER = '/content/drive/MyDrive/job-offer-extractor/adapter_v1'
model.save_pretrained(ADAPTER)
tokenizer.save_pretrained(ADAPTER)

REPO = 'mayss1/qwen2.5-1.5b-offres-tn-lora'
model.push_to_hub(REPO, private=True)
tokenizer.push_to_hub(REPO, private=True)
print('Adaptateur sauvegardé sur Drive et sur', REPO)

In [ ]:
GEN_KWARGS = {}

@torch.no_grad()
def extraire(offre, max_new_tokens=900):
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': offre}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors='pt', return_dict=True
    ).to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id, **GEN_KWARGS)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

In [ ]:
GEN_KWARGS = {'repetition_penalty': 1.05}
res_rp105 = evaluer('finetune_v1_rp1.05')

GEN_KWARGS = {'repetition_penalty': 1.1}
res_rp110 = evaluer('finetune_v1_rp1.1')

comparaison = pd.concat([res_base, res_ft, res_rp105, res_rp110], axis=1)
comparaison.to_csv(f'{RESULTATS}/comparaison_v1_decodage.csv')
comparaison